In [1]:
import gradio as gr
import numpy as np
import joblib
import shap
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

# ── Load model and scaler ────────────────────────────────────────
base      = r"C:\Users\abdul\Amr senior\dataset"
model     = joblib.load(f"{base}\\best_model_xgb.pkl")
scaler    = joblib.load(f"{base}\\scaler.pkl")
explainer = shap.TreeExplainer(model)

FEATURE_NAMES = [
    'Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
    'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age',
    'Glucose_BMI', 'Age_BMI', 'Glucose_squared', 'Insulin_BMI'
]

# ── Prediction function ──────────────────────────────────────────
def predict(pregnancies, glucose, blood_pressure, skin_thickness,
            insulin, bmi, pedigree, age):

    # Feature engineering (same as training)
    glucose_bmi     = glucose * bmi
    age_bmi         = age * bmi
    glucose_squared = glucose ** 2
    insulin_bmi     = insulin * bmi

    raw = np.array([[pregnancies, glucose, blood_pressure, skin_thickness,
                     insulin, bmi, pedigree, age,
                     glucose_bmi, age_bmi, glucose_squared, insulin_bmi]])

    # Scale
    scaled = scaler.transform(raw)

    # Predict
    proba     = model.predict_proba(scaled)[0][1]
    threshold = 0.40
    label     = "🔴 Diabetic" if proba >= threshold else "🟢 Non-Diabetic"
    risk      = "High" if proba >= 0.70 else "Moderate" if proba >= 0.40 else "Low"

    result = (
        f"Prediction   : {label}\n"
        f"Probability  : {proba:.1%}\n"
        f"Risk Level   : {risk}\n"
        f"Threshold    : {threshold}"
    )

    # ── SHAP force plot ──────────────────────────────────────────
    shap_values = explainer.shap_values(scaled)

    fig, ax = plt.subplots(figsize=(10, 3))
    shap.plots._waterfall.waterfall_legacy(
        explainer.expected_value,
        shap_values[0],
        feature_names=FEATURE_NAMES,
        max_display=10,
        show=False
    )
    plt.title("Feature contributions for this patient", fontsize=12, pad=10)
    plt.tight_layout()

    return result, fig


# ── Gradio Interface ─────────────────────────────────────────────
with gr.Blocks(title="Diabetes Classifier", theme=gr.themes.Soft()) as app:

    gr.Markdown("""
    # 🩺 Diabetes Risk Classifier
    Enter the patient's lab values below to get an instant prediction
    with a full explanation of which factors influenced the result.
    ---
    """)

    with gr.Row():
        with gr.Column():
            gr.Markdown("### 📋 Patient Lab Values")

            pregnancies    = gr.Slider(0, 17, value=1, step=1,
                                       label="Pregnancies")
            glucose        = gr.Slider(50, 250, value=110, step=1,
                                       label="Glucose (mg/dL)")
            blood_pressure = gr.Slider(40, 130, value=72, step=1,
                                       label="Blood Pressure (mm Hg)")
            skin_thickness = gr.Slider(0, 100, value=20, step=1,
                                       label="Skin Thickness (mm)")
            insulin        = gr.Slider(0, 900, value=80, step=1,
                                       label="Insulin (mu U/ml)")
            bmi            = gr.Slider(10.0, 70.0, value=25.0, step=0.1,
                                       label="BMI")
            pedigree       = gr.Slider(0.0, 2.5, value=0.35, step=0.01,
                                       label="Diabetes Pedigree Function")
            age            = gr.Slider(18, 90, value=30, step=1,
                                       label="Age")

            btn = gr.Button("🔍 Analyze Patient", variant="primary")

        with gr.Column():
            gr.Markdown("### 📊 Prediction Result")
            result_box = gr.Textbox(
                label="Result",
                lines=5,
                interactive=False
            )
            gr.Markdown("### 🔬 SHAP Explanation")
            shap_plot = gr.Plot(label="Feature Contributions")

    btn.click(
        fn=predict,
        inputs=[pregnancies, glucose, blood_pressure, skin_thickness,
                insulin, bmi, pedigree, age],
        outputs=[result_box, shap_plot]
    )

    gr.Markdown("""
    ---
    > ⚠️ **Disclaimer:** This tool is for research purposes only.
    Always consult a licensed physician for medical decisions.
    """)

# ── Launch ───────────────────────────────────────────────────────
app.launch(share=True)

C:\Users\abdul\AppData\Local\Temp\ipykernel_16896\3693021451.py:69: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title="Diabetes Classifier", theme=gr.themes.Soft()) as app:


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://84bd200203e7017d5c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


C:\Users\abdul\anaconda3\envs\ocr\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\abdul\anaconda3\envs\ocr\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [2]:
import requests

response = requests.get("http://127.0.0.1:5000/health")
print(response.json())

{'message': 'Model and scaler loaded successfully', 'model': 'XGBoost', 'status': 'ok', 'version': '1.0.0'}


In [3]:
import requests
import json

base = "http://127.0.0.1:5000"

patient = {
    "pregnancies": 2,
    "glucose": 148,
    "blood_pressure": 72,
    "skin_thickness": 35,
    "insulin": 150,
    "bmi": 33.6,
    "diabetes_pedigree_function": 0.627,
    "age": 50
}

# ── Test 1: /predict ─────────────────────────────────────────────
r1 = requests.post(f"{base}/predict", json=patient)
print("✅ /predict:", r1.json())

# ── Test 2: /predict/explain ─────────────────────────────────────
r2 = requests.post(f"{base}/predict/explain", json=patient)
print("\n✅ /predict/explain:", json.dumps(r2.json(), indent=2))

# ── Test 3: /predict/batch ───────────────────────────────────────
r3 = requests.post(f"{base}/predict/batch", json={"patients": [patient, patient]})
print("\n✅ /predict/batch:", json.dumps(r3.json(), indent=2))

✅ /predict: {'input': {'age': 50.0, 'blood_pressure': 72.0, 'bmi': 33.6, 'diabetes_pedigree_function': 0.627, 'glucose': 148.0, 'insulin': 150.0, 'pregnancies': 2.0, 'skin_thickness': 35.0}, 'prediction': 'Diabetic', 'probability': 0.9389, 'risk_level': 'High', 'threshold': 0.4}


JSONDecodeError: Expecting value: line 1 column 1 (char 0)